# Chunking Practical: Most Useful Methods
This notebook covers the following methods:

1. `RecursiveCharacterTextSplitter`
2. `TokenTextSplitter`
3. `CharacterTextSplitter`
4. Fixed-size chunking
5. Content-aware chunking
6. Document structure-based chunking
7. Semantic chunking

## Learning objectives

- Explain chunk size and chunk overlap
- Compare fixed, character, recursive, token, structural, and semantic splitting
- Understand why different strategies produce different boundaries
- Choose a suitable starting method for a given document type
- Evaluate chunks instead of blindly selecting one splitter


1. Fixed-size character chunking
   → CharacterTextSplitter

2. Fixed-size token chunking
   → TokenTextSplitter

3. Recursive structure-aware chunking
   → RecursiveCharacterTextSplitter

4. Document structure-based chunking
   → MarkdownHeaderTextSplitter, HTML splitters, etc.

5. Semantic chunking
   → Embedding-based topic boundaries

CharacterTextSplitter

→ Primarily separator-based

→ Length measured using characters

TokenTextSplitter

→ Directly splits according to tokens

RecursiveCharacterTextSplitter

→ Tries to preserve paragraphs, lines and words

→ Not purely blind fixed-size chunking

In [50]:
from __future__ import annotations
import math
import re
from typing import Any
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from langchain_text_splitters import (
    CharacterTextSplitter,
    MarkdownHeaderTextSplitter,
    RecursiveCharacterTextSplitter,
    TokenTextSplitter,
)

# 1. Sample Document

The sample document intentionally contains multiple topics and Markdown headings.  
This makes it useful for comparing basic, structure-aware, and semantic chunking.

In [51]:
document = """
# Artificial Intelligence

Artificial intelligence enables machines to perform tasks that normally require human intelligence.
AI systems can analyze information, identify patterns, and support decision-making.

## Machine Learning

Machine learning is a branch of artificial intelligence.
It enables systems to learn patterns from historical data instead of relying only on explicitly programmed rules.
Supervised learning uses labelled examples, while unsupervised learning discovers patterns in unlabelled data.

## Deep Learning

Deep learning uses neural networks containing multiple layers.
It is commonly applied to image recognition, speech processing, and natural-language applications.

# Employee Leave Policy

Employees receive twenty paid leave days during a calendar year.
Leave requests should be submitted to the reporting manager before the planned absence.
Emergency leave may be submitted later when advance notice is not possible.

## Approval Process

The manager reviews the leave balance, project schedule, and team availability.
Approved leave is recorded in the human-resource management system.

# Information Security

Employees must not share passwords or authentication codes.
Sensitive company information should be stored only in approved systems.
Suspected security incidents must be reported to the security team immediately.
""".strip()

In [52]:
print(document)

# Artificial Intelligence

Artificial intelligence enables machines to perform tasks that normally require human intelligence.
AI systems can analyze information, identify patterns, and support decision-making.

## Machine Learning

Machine learning is a branch of artificial intelligence.
It enables systems to learn patterns from historical data instead of relying only on explicitly programmed rules.
Supervised learning uses labelled examples, while unsupervised learning discovers patterns in unlabelled data.

## Deep Learning

Deep learning uses neural networks containing multiple layers.
It is commonly applied to image recognition, speech processing, and natural-language applications.

# Employee Leave Policy

Employees receive twenty paid leave days during a calendar year.
Leave requests should be submitted to the reporting manager before the planned absence.
Emergency leave may be submitted later when advance notice is not possible.

## Approval Process

The manager reviews the lea

In [53]:
from pypdf import PdfReader
reader = PdfReader("/Users/sainaveenvasa/Downloads/The Beginning of Infinity Explanations that Transform the World.pdf")
page_texts = []
for page_number, page in enumerate(reader.pages, start=1):
    text = page.extract_text() or ""

    if text.strip():
        page_texts.append(text)

    print(
        f"Page {page_number}: "
        f"{len(text)} characters extracted"
    )

Page 1: 0 characters extracted
Page 2: 84 characters extracted
Page 3: 0 characters extracted
Page 4: 25 characters extracted
Page 5: 0 characters extracted
Page 6: 84 characters extracted
Page 7: 2609 characters extracted
Page 8: 603 characters extracted
Page 9: 669 characters extracted
Page 10: 1739 characters extracted
Page 11: 573 characters extracted
Page 12: 1685 characters extracted
Page 13: 2649 characters extracted
Page 14: 2421 characters extracted
Page 15: 2200 characters extracted
Page 16: 2569 characters extracted
Page 17: 2492 characters extracted
Page 18: 2611 characters extracted
Page 19: 2525 characters extracted
Page 20: 2569 characters extracted
Page 21: 2461 characters extracted
Page 22: 2514 characters extracted
Page 23: 2522 characters extracted
Page 24: 2538 characters extracted
Page 25: 2538 characters extracted
Page 26: 2600 characters extracted
Page 27: 2546 characters extracted
Page 28: 2461 characters extracted
Page 29: 2608 characters extracted
Page 30: 243

In [54]:
document = "\n\n".join(page_texts)

In [55]:
print(document)

david deutsch
The Beginning of Infinity
Explanations that Transform the World
VIKING

The Beginning of Infinity

david deutsch
The Beginning of Infinity
Explanations that Transform the World
VIKING

VIKING
Published by the Penguin Group
Penguin Group (USA) Inc., 375 Hudson Street,
New York, New York 10014, U.S.A.
Penguin Group (Canada), 90 Eglinton Avenue East, Suite 700, 
Toronto, Ontario, Canada M4P 2Y3 
(a division of Pearson Penguin Canada Inc.) 
Penguin Books Ltd, 80 Strand, London WC2R 0RL, England
Penguin Ireland, 25 St. Stephen’s Green, Dublin 2, Ireland 
(a division of Penguin Books Ltd)
Penguin Books Australia Ltd, 250 Camberwell Road, Camberwell,
Victoria 3124, Australia
(a division of Pearson Australia Group Pty Ltd) 
Penguin Books India Pvt Ltd, 11 Community Centre, Panchsheel Park,
New Delhi – 110 017, India
Penguin Group (NZ), 67 Apollo Drive, Rosedale, Auckland 0632, 
New Zealand (a division of Pearson New Zealand Ltd)
Penguin Books (South Africa) (Pty) Ltd, 24 Sturdee 

In [56]:
# 2. Helper Functions

# These helpers print chunks consistently and collect simple statistics.

from typing import Any

def print_chunks(
    title: str,
    chunks: list[Any],
    preview_limit: int = 500,
) -> None:
    print(f"\n{'=' * 90}")
    print(title)
    print(f"Total chunks: {len(chunks)}")
    print("=" * 90)

    for index, chunk in enumerate(chunks, start=1):

        # Get text from a LangChain Document or normal Python object
        text = getattr(
            chunk,
            "page_content",
            str(chunk),
        )

        # Get metadata if available
        metadata = getattr(
            chunk,
            "metadata",
            {},
        )

        print(
            f"\nChunk {index} | "
            f"characters={len(text)}"
        )

        if metadata:
            print("Metadata:", metadata)

        if len(text) > preview_limit:
            print(text[:preview_limit] + "...")
        else:
            print(text)

In [57]:
from typing import Any
import numpy as np


def chunk_stats(
    method: str,
    chunks: list[Any],
) -> dict[str, Any]:

    lengths = [
        len(
            getattr(
                chunk,
                "page_content",
                str(chunk),
            )
        )
        for chunk in chunks
    ]

    return {
        "method": method,
        "number_of_chunks": len(chunks),
        "minimum_length": min(lengths) if lengths else 0,
        "maximum_length": max(lengths) if lengths else 0,
        "average_length": (
            round(float(np.mean(lengths)), 2)
            if lengths
            else 0
        ),
        "total_stored_characters": sum(lengths),
    }

# 3. Fixed-Size Chunking

Fixed-size chunking divides content after a specified number of characters or tokens.

It is simple and predictable, but it may break:

- Words
- Sentences
- Paragraphs

In [58]:
def fixed_size_chunking(text: str, chunk_size: int, chunk_overlap: int = 0) -> list[str]:
    if chunk_size <= 0:
        raise ValueError("chunk_size must be greater than zero.")

    if chunk_overlap < 0 or chunk_overlap >= chunk_size:
        raise ValueError(
            "chunk_overlap must satisfy 0 <= chunk_overlap < chunk_size."
        )

    step = chunk_size - chunk_overlap

    return [
        text[start:start + chunk_size]
        for start in range(0, len(text), step)
    ]

In [59]:
text = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"

In [60]:
len(text)

26

In [61]:
fixed_size_chunking(text, chunk_size=10, chunk_overlap=2)

['ABCDEFGHIJ', 'IJKLMNOPQR', 'QRSTUVWXYZ', 'YZ']

10 - 2 = 8

So chunks begin at positions 0, 8, 16, 24.

In [62]:
document

'david deutsch\nThe Beginning of Infinity\nExplanations that Transform the World\nVIKING\n\nThe Beginning of Infinity\n\ndavid deutsch\nThe Beginning of Infinity\nExplanations that Transform the World\nVIKING\n\nVIKING\nPublished by the Penguin Group\nPenguin Group (USA) Inc., 375 Hudson Street,\nNew York, New York 10014, U.S.A.\nPenguin Group (Canada), 90 Eglinton Avenue East, Suite 700, \nToronto, Ontario, Canada M4P 2Y3 \n(a division of Pearson Penguin Canada Inc.) \nPenguin Books Ltd, 80 Strand, London WC2R 0RL, England\nPenguin Ireland, 25 St. Stephen’s Green, Dublin 2, Ireland \n(a division of Penguin Books Ltd)\nPenguin Books Australia Ltd, 250 Camberwell Road, Camberwell,\nVictoria 3124, Australia\n(a division of Pearson Australia Group Pty Ltd) \nPenguin Books India Pvt Ltd, 11 Community Centre, Panchsheel Park,\nNew Delhi – 110 017, India\nPenguin Group (NZ), 67 Apollo Drive, Rosedale, Auckland 0632, \nNew Zealand (a division of Pearson New Zealand Ltd)\nPenguin Books (South 

In [63]:
fixed_chunks=fixed_size_chunking(text=document, chunk_size=700, chunk_overlap=50)

In [64]:
fixed_chunks

['david deutsch\nThe Beginning of Infinity\nExplanations that Transform the World\nVIKING\n\nThe Beginning of Infinity\n\ndavid deutsch\nThe Beginning of Infinity\nExplanations that Transform the World\nVIKING\n\nVIKING\nPublished by the Penguin Group\nPenguin Group (USA) Inc., 375 Hudson Street,\nNew York, New York 10014, U.S.A.\nPenguin Group (Canada), 90 Eglinton Avenue East, Suite 700, \nToronto, Ontario, Canada M4P 2Y3 \n(a division of Pearson Penguin Canada Inc.) \nPenguin Books Ltd, 80 Strand, London WC2R 0RL, England\nPenguin Ireland, 25 St. Stephen’s Green, Dublin 2, Ireland \n(a division of Penguin Books Ltd)\nPenguin Books Australia Ltd, 250 Camberwell Road, Camberwell,\nVictoria 3124, Australia\n(a divisi',
 'ad, Camberwell,\nVictoria 3124, Australia\n(a division of Pearson Australia Group Pty Ltd) \nPenguin Books India Pvt Ltd, 11 Community Centre, Panchsheel Park,\nNew Delhi – 110 017, India\nPenguin Group (NZ), 67 Apollo Drive, Rosedale, Auckland 0632, \nNew Zealand (a d

In [65]:
len(fixed_chunks)


1763

In [66]:
print_chunks("Manual Fixed-Size Chunking", fixed_chunks)


Manual Fixed-Size Chunking
Total chunks: 1763

Chunk 1 | characters=700
david deutsch
The Beginning of Infinity
Explanations that Transform the World
VIKING

The Beginning of Infinity

david deutsch
The Beginning of Infinity
Explanations that Transform the World
VIKING

VIKING
Published by the Penguin Group
Penguin Group (USA) Inc., 375 Hudson Street,
New York, New York 10014, U.S.A.
Penguin Group (Canada), 90 Eglinton Avenue East, Suite 700, 
Toronto, Ontario, Canada M4P 2Y3 
(a division of Pearson Penguin Canada Inc.) 
Penguin Books Ltd, 80 Strand, London WC2R 0RL...

Chunk 2 | characters=700
ad, Camberwell,
Victoria 3124, Australia
(a division of Pearson Australia Group Pty Ltd) 
Penguin Books India Pvt Ltd, 11 Community Centre, Panchsheel Park,
New Delhi – 110 017, India
Penguin Group (NZ), 67 Apollo Drive, Rosedale, Auckland 0632, 
New Zealand (a division of Pearson New Zealand Ltd)
Penguin Books (South Africa) (Pty) Ltd, 24 Sturdee Avenue,
Rosebank, Johannesburg 2196, South Afri

## What to observe

- Every chunk targets the same maximum character size.
- Fifty characters are repeated between neighbouring chunks.
- A boundary may occur in the middle of a sentence or heading.
- `Overlap protects some boundary information but also creates duplication`

# 4. `CharacterTextSplitter`

`CharacterTextSplitter` splits using one selected separator.

In this example, the separator is a blank line (`"\n\n"`), so it tries to keep paragraphs together.

This is easy to understand, but one separator may not be sufficient for every document.

In [67]:
from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter
)

In [68]:
character_splitter=CharacterTextSplitter(chunk_size=300, chunk_overlap=50, separator="\n\n",length_function=len)

In [69]:
character_chunks=character_splitter.split_text(document)

Created a chunk of size 2609, which is longer than the specified 300
Created a chunk of size 603, which is longer than the specified 300
Created a chunk of size 669, which is longer than the specified 300
Created a chunk of size 1739, which is longer than the specified 300
Created a chunk of size 573, which is longer than the specified 300
Created a chunk of size 1685, which is longer than the specified 300
Created a chunk of size 2649, which is longer than the specified 300
Created a chunk of size 2421, which is longer than the specified 300
Created a chunk of size 2200, which is longer than the specified 300
Created a chunk of size 2569, which is longer than the specified 300
Created a chunk of size 2492, which is longer than the specified 300
Created a chunk of size 2611, which is longer than the specified 300
Created a chunk of size 2525, which is longer than the specified 300
Created a chunk of size 2569, which is longer than the specified 300
Created a chunk of size 2461, which i

In [70]:
print_chunks("CharacterTextSplitter", character_chunks)


CharacterTextSplitter
Total chunks: 492

Chunk 1 | characters=197
david deutsch
The Beginning of Infinity
Explanations that Transform the World
VIKING

The Beginning of Infinity

david deutsch
The Beginning of Infinity
Explanations that Transform the World
VIKING

Chunk 2 | characters=2608
VIKING
Published by the Penguin Group
Penguin Group (USA) Inc., 375 Hudson Street,
New York, New York 10014, U.S.A.
Penguin Group (Canada), 90 Eglinton Avenue East, Suite 700, 
Toronto, Ontario, Canada M4P 2Y3 
(a division of Pearson Penguin Canada Inc.) 
Penguin Books Ltd, 80 Strand, London WC2R 0RL, England
Penguin Ireland, 25 St. Stephen’s Green, Dublin 2, Ireland 
(a division of Penguin Books Ltd)
Penguin Books Australia Ltd, 250 Camberwell Road, Camberwell,
Victoria 3124, Australia
(a divis...

Chunk 3 | characters=603
Contents
  Acknowledgements  vi
  Introduction  vii
 
 1. The Reach of Explanations 1
  2. Closer to Reality 34
  3. The Spark  42
  4. Creation 78
  5. The Reality of Abstractio

## Important observation

`CharacterTextSplitter` is separator-driven. If an individual separator-delimited block is already larger than the target size, the output may not behave like strict fixed-width slicing.

Use it when the source has a predictable separator and you want a simple baseline.

# 5. `RecursiveCharacterTextSplitter`

This splitter tries separators recursively.

Typical order:

```text
Paragraph → line → sentence-like boundary → word → character
```

It first tries to preserve large natural units. If a unit is too large, it moves to a finer separator.

In [71]:
recursive_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", ". ", " ", ""],
    chunk_size=300,
    chunk_overlap=50,
    length_function=len,
    add_start_index=True,
)

In [72]:
recursive_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", ". ", " ", ""],
    chunk_size=5,
    chunk_overlap=2,
    length_function=len,
    add_start_index=True,
)

In [73]:
text = """Artificial Intelligence

AI enables machines to perform intelligent tasks.
Machine learning is a part of AI.

Employee Leave Policy

Employees receive 20 paid leave days.
Leave must be approved by the manager.
"""

In [74]:
recursive_splitter.create_documents(
    texts=[text],
    metadatas=[{"source": "classroom_sample.md"}],
)

[Document(metadata={'source': 'classroom_sample.md', 'start_index': 0}, page_content='Artif'),
 Document(metadata={'source': 'classroom_sample.md', 'start_index': 3}, page_content='ifici'),
 Document(metadata={'source': 'classroom_sample.md', 'start_index': 6}, page_content='cial'),
 Document(metadata={'source': 'classroom_sample.md', 'start_index': 11}, page_content='Inte'),
 Document(metadata={'source': 'classroom_sample.md', 'start_index': 13}, page_content='telli'),
 Document(metadata={'source': 'classroom_sample.md', 'start_index': 16}, page_content='ligen'),
 Document(metadata={'source': 'classroom_sample.md', 'start_index': 19}, page_content='ence'),
 Document(metadata={'source': 'classroom_sample.md', 'start_index': 25}, page_content='AI'),
 Document(metadata={'source': 'classroom_sample.md', 'start_index': 28}, page_content='enab'),
 Document(metadata={'source': 'classroom_sample.md', 'start_index': 30}, page_content='ables'),
 Document(metadata={'source': 'classroom_sample.md